In [20]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import importlib


## config

In [21]:
# ===== USER SETTINGS =====
FOLDER_A = Path("outputs/2_fairness/LNS/1000_gini")
FOLDER_B = Path("outputs/2_fairness/LNS/1000_minmax")
FOLDER_BASE = Path("outputs/2_fairness/LNS/1000_jain")

LABEL_A = "Gini"
LABEL_B = "MinMax"
LABEL_BASE = "Jain"

OUTPUT_DIR = Path("outputs/2_fairness/LNS/plots")
OUTPUT_DIR.mkdir(exist_ok=True)


In [22]:
import eval_fairness
import importlib
import pandas as pd

importlib.reload(eval_fairness)

evaluate_folder = eval_fairness.evaluate_folder

df_a = pd.DataFrame(eval_fairness.evaluate_folder(FOLDER_A))
df_b = pd.DataFrame(eval_fairness.evaluate_folder(FOLDER_B))
df_base = pd.DataFrame(eval_fairness.evaluate_folder(FOLDER_BASE))

df_a["method"] = LABEL_A
df_b["method"] = LABEL_B
df_base["method"] = LABEL_BASE

common = (
    set(df_a["instance"])
    & set(df_b["instance"])
    & set(df_base["instance"])
)

df = pd.concat([
    df_a[df_a["instance"].isin(common)],
    df_b[df_b["instance"].isin(common)],
    df_base[df_base["instance"].isin(common)],
], ignore_index=True)

print(f"Common instances: {len(common)}")


Common instances: 30


In [23]:
plt.figure(figsize=(6, 4))

df.boxplot(
    column="total_cost",
    by="method",
    grid=False,
)

plt.xlabel("Fairness objective")
plt.ylabel("Total route cost")
plt.title("Cost distribution by fairness objective")
plt.suptitle("")
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.savefig(OUTPUT_DIR / "cost_by_fairness_boxplot.png",
            dpi=300, bbox_inches="tight")
plt.close()


<Figure size 600x400 with 0 Axes>

In [ ]:
plt.figure(figsize=(6, 4))

for method, dfi in df.groupby("method"):
    plt.scatter(
        dfi["total_cost"],
        dfi["fairness"],
        label=method,
        alpha=0.6,
        s=30
    )

plt.xlabel("Total cost")
plt.ylabel("Fairness")
plt.title("Cost-fairness trade-off")
plt.legend()
plt.grid(True)

plt.savefig(OUTPUT_DIR / "cost_fairness_tradeoff_all.png",
            dpi=300, bbox_inches="tight")
plt.close()


In [25]:
plt.figure(figsize=(6, 4))

for method, dfi in df.groupby("method"):
    plt.scatter(
        dfi["fairness_weight"],
        dfi["total_cost"],
        label=method,
        alpha=0.6,
        s=30
    )

plt.xlabel("Fairness weight")
plt.ylabel("Total cost")
plt.title("Impact of fairness preference on solution cost")
plt.legend()
plt.grid(True)

plt.savefig(OUTPUT_DIR / "cost_vs_weight_all.png",
            dpi=300, bbox_inches="tight")
plt.close()


In [27]:
from scipy.stats import friedmanchisquare

pivot_cost = df.pivot(
    index="instance",
    columns="method",
    values="total_cost"
)

stat, p = friedmanchisquare(
    *[pivot_cost[c] for c in pivot_cost.columns]
)

print(f"Friedman test (cost): statistic={stat:.3f}, p-value={p:.4e}")


Friedman test (cost): statistic=44.067, p-value=2.6980e-10


In [28]:
from scipy.stats import wilcoxon
from itertools import combinations

alpha = 0.05
methods = pivot_cost.columns
m = len(methods)
bonferroni = alpha / (m * (m - 1) / 2)

print(f"Bonferroni-corrected alpha = {bonferroni:.4f}\n")

for a, b in combinations(methods, 2):
    stat, p = wilcoxon(pivot_cost[a], pivot_cost[b])
    sig = "YES" if p < bonferroni else "NO"
    print(f"{a} vs {b}: p={p:.4e}  significant={sig}")


Bonferroni-corrected alpha = 0.0167

Gini vs Jain: p=1.8626e-09  significant=YES
Gini vs MinMax: p=7.6121e-03  significant=YES
Jain vs MinMax: p=3.7253e-09  significant=YES


In [29]:
from scipy.stats import spearmanr

baseline = "Jain"
pivot_cost = df.pivot(index="instance", columns="method", values="total_cost")
weights = df.groupby("instance")["fairness_weight"].first()

for method in pivot_cost.columns:
    if method == baseline:
        continue

    ratio = pivot_cost[method] / pivot_cost[baseline]
    rho, p = spearmanr(weights, ratio)

    print(
        f"{method} vs {baseline}: "
        f"Spearman rho={rho:.3f}, p={p:.4e}"
    )


Gini vs Jain: Spearman rho=-0.248, p=1.8663e-01
MinMax vs Jain: Spearman rho=-0.069, p=7.1898e-01


In [30]:
pivot_time = df.pivot(
    index="instance",
    columns="method",
    values="execution_time"
)

plt.figure(figsize=(6, 4))

for method in pivot_time.columns:
    plt.scatter(
        weights,
        pivot_time[method],
        label=method,
        alpha=0.6,
        s=30
    )

plt.yscale("log")
plt.xlabel("Fairness weight")
plt.ylabel("Execution time (log scale)")
plt.title("Runtime sensitivity to fairness preference")
plt.legend()
plt.grid(True)

plt.savefig(OUTPUT_DIR / "runtime_vs_weight_log.png",
            dpi=300, bbox_inches="tight")
plt.close()


In [31]:
ranks = pivot_cost.rank(axis=1)

plt.figure(figsize=(6, 4))

for method in ranks.columns:
    plt.plot(
        ranks[method].values,
        label=method,
        alpha=0.7
    )

plt.xlabel("Instance index")
plt.ylabel("Cost rank (lower = cheaper)")
plt.title("Stability of instance ranking across fairness objectives")
plt.legend()
plt.grid(True)

plt.savefig(OUTPUT_DIR / "rank_stability.png",
            dpi=300, bbox_inches="tight")
plt.close()
